# Spark Application Logs
A proper application must have some kind of application logging.

## How to use Log4j with pyspark?
Configuring Log4j is a three step process :
- Create a Log4j configuration file.
- Configure Spark JVM to pickup the Log4j configuration file.
- Create a python class to get Spark's Log4j instance and use it in the pyspark program

## Log4j properties file
Log4j works almost the same way as python logs.
### Componenets of Log4j : 
- Logger : It is a set of apis which we are going to use in our application.
- Configurations : This is defined in the Log4j properties file and they are loaded with the loggers at run time.
    - Log4j configurations are defined in the hierarchy and the top most hierarchy is the root category.
        - Example : 
        ```python
            # Root logger configuration
            log4j.rootCategory=INFO, console
        ```
            - Here INFO is the log level
    - For any hierarchy or category we define two things first is the log level and the second thing is the list of appenders.
    - Log4j supports multiple log levels such as INFO,DEBUG,WARN,ERROR
    - Here in the top most level info will be shown because we have set INFO to be the root log level.
    - The root logger is the default logger that all other loggers inherit from.
    - If you don’t define a more specific logger (like log4j.logger.org.apache.spark), the root logger decides what happens to log messages.
    - It controls what level of messages are logged and where they go.
- Appender : Appenders are the output destinations such as console and log file. These appenders are also configured in the Log4j property file.
    - Console appender : 
        - Example : 
        ```python
            # Root logger configuration
            log4j.rootCategory=INFO, console
        ```
            - Here console is the appender
- **NOTE :** Hierarchy and Appender section define the root level Log4j configurations and they will stop all the log messages sent by the spark and other packages except Warning and errors.

#### Automatically detect the configuration file and set up the configuration for Log4j for a spark program written in python
In this pyspark application with dynamic project level log4j logging configured here in this example is portable in nature because the log4j.properties file along with the logs folder stays inside the project's folder which makes it extremely portable

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand

# For keeping track of time taken to complete the spark computation task
import time
import sys

# spark Log4j logging related imports
from pyspark import SparkConf
import os

# Logging related Spark configurations setup
# log4k.properties configuration file path setup
# Determine project directory — works in both script & notebook
try:
    project_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined in interactive mode (e.g., Jupyter)
    project_dir = os.getcwd()

# Dynamically define your log directory (for example: logs inside project)
log_dir = os.path.join(project_dir, "log4j_properties", "logs")
os.makedirs(log_dir, exist_ok=True)  # ensure it exists

log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")

# Create SparkConf with custom Log4j config
conf = (
    SparkConf()
    .setAppName("CPU_Stress_Test")
    .setMaster("local[*]")
    # JVM property for log4j 
    # Use this Is you have set the directory where the log file must be generated inside the log4j.properties file
    # .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
    # .set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
    # Use this If you want to setup the directory where the Log files must be generated using python
    .set("spark.driver.extraJavaOptions",
         f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
    .set("spark.executor.extraJavaOptions",
         f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
    # Ensure executors also get it via --files equivalent
    .set("spark.files", log4j_config_path)
)

start = time.time()

#Create Spark session in local mode using all cores
spark = SparkSession.builder \
    .config(conf=conf)\
    .getOrCreate()

print("Spark master:", spark.sparkContext.master)
print("Total cores Spark sees:", spark.sparkContext.defaultParallelism)

#Create a large synthetic dataset (e.g., 100 million rows)
num_rows = 99_999
num_partitions = spark.sparkContext.defaultParallelism  # same as CPU cores

df = spark.range(0, num_rows, numPartitions=num_partitions) \
           .withColumn("random_val", rand())

# I want to know the amount of Ram taken by the dataFrame in Mbs
# converting df to rdd rows
rdd = df.rdd.map(lambda row: row.asDict())
# Estimate memory size of one partition
def estimate_partition_size(partition):
    import sys
    size = 0
    for record in partition:
        size += sys.getsizeof(record)
    yield size

partition_sizes = rdd.mapPartitions(estimate_partition_size).collect()
total_bytes = sum(partition_sizes)
total_mb = total_bytes / (1024 * 1024)

#Apply heavy transformations — wide operations
#    Force Spark to use multiple stages and shuffles
aggregated_df = (
    df.withColumn("squared", col("random_val") * col("random_val"))
      .groupBy((col("id") % 100).alias("group"))  # 100 groups
      .avg("squared")                            # aggregation
      .orderBy("group")                          # shuffle operation
)

#Trigger computation (action)
aggregated_df.show()

end = time.time()

time_taken_in_sec = end - start
time_taken_in_min = (end - start) / 60
print(f"""
        Spark task complete!
        Time taken in seconds = {time_taken_in_sec}
        Time taken in minutes = {time_taken_in_min}
        Estimated DataFrame size in memory: {total_mb:.2f} MB
""")


Setting default log level to "DEBUG".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/10/29 12:18:16 INFO Server: jetty-11.0.24; built: 2024-08-26T18:11:22.448Z; git: 5dfc59a691b748796f922208956bd1f2794bcd16; jvm 17.0.15+6-Ubuntu-0ubuntu120.04
25/10/29 12:18:16 INFO Server: Started Server@73b4d8b7{STARTING}[11.0.24,sto=30000] @2048ms
25/10/29 12:18:16 INFO AbstractConnector: Started ServerConnector@3f1256c8{HTTP/1.1, (http/1.1)}{0.0.0.0:4040}
25/10/29 12:18:16 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@10d377b4{/,null,AVAILABLE,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Stopped o.s.j.s.ServletContextHandler@10d377b4{/,null,STOPPED,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@16a4e6e3{/jobs,null,AVAILABLE,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@2f0d9e56{/jobs/json,null,AVAILABLE,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@4bc7147{/jobs/job,null,AVAILABLE,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Started o.s.j.s.ServletC

+-----+-------------------+
|group|       avg(squared)|
+-----+-------------------+
|    0| 0.3279938617519789|
|    1|0.34576483889045606|
|    2|0.32681397541072815|
|    3| 0.3446750728815876|
|    4| 0.3343884808867916|
|    5|0.32564363285727255|
|    6|0.34520534230646555|
|    7|0.34078275580841333|
|    8|0.32975440803719874|
|    9|0.33009214531352643|
|   10| 0.3391300519163157|
|   11| 0.3297460270684851|
|   12|0.34306169008989895|
|   13|0.34458272746086255|
|   14|0.32444943685993444|
|   15|0.33191574922919065|
|   16|0.31318415768603897|
|   17|0.35047172118867703|
|   18|0.33270819405072327|
|   19| 0.3238464271903375|
+-----+-------------------+
only showing top 20 rows

        Spark task complete!
        Time taken in seconds = 7.792145490646362
        Time taken in minutes = 0.1298690915107727
        Estimated DataFrame size in memory: 17.55 MB



### Explaination
- ```log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")```
    - This constructs the absolute path to my Log4j configuration file that lives inside my project directory.
    - if the project tree looks like this 
    ```bash
        pyspark/Spark_programming_model/
        ├── log4j_properties/
        │   └── log4j.properties
        └── your_spark_script.py

    ```
    - Then the log4j_config_path will end_up as something like this ```/home/aditya/pyspark/Spark_programming_model/log4j_properties/log4j.properties```
    - I need the full path because the JVM (which Spark runs on) must know where the log4j file is located.
- ```python
        conf = (
            SparkConf()
            .setAppName("CPU_Stress_Test")
            .setMaster("local[*]")
            # JVM property for log4j v1
            .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
            .set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
            # Ensure executors also get it via --files equivalent
            .set("spark.files", log4j_config_path)
        )
    ```
    - ```SparkConf()```
        - SparkConf creates a Spark configuration object that stores key-value pairs Spark uses when starting up.
        - I will have to pass this to the SparkSession so that Spark knows how to start my local cluster
    - ```.set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")```
        - This is the cruicial part for logging. I am passing a JVM system property (-D...) into Spark's driver process
        - ```spark.driver.extraJavaOptions``` : extra Java options for the Spark Driver JVM
        - ```-Dlog4j.configuration=file:/path/to/log4j.properties``` : tells Log4j where my custom config file is stored
        - Essentially:
            - “Hey Spark driver, when you start your JVM, use this custom Log4j properties file instead of the default one.”
        - **NOTE:** 
            - If you want to set the Log file directory from your Spark application using python then you will have to make changes 
            ```python
                .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
            ```
            to
            ```python
                .set("spark.driver.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
            ```
            you will have to set the path to the log_dir like this dynamically
            ```python
                log_dir = os.path.join(project_dir, "log4j_properties", "logs")
                os.makedirs(log_dir, exist_ok=True)  # ensure it exists
            ```
- ```.set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")```
    - Same idea — but applies to executor JVMs (the parallel worker processes that actually execute your tasks).
    - This ensures:
        - Driver logs → follow your log4j config
        - Executor logs → also follow your log4j config
    - Without this line, only your driver would use your custom Log4j configuration, and executor logs might still use the default Spark log setup.
    - **NOTE:** 
        - If you want to set the Log file directory from your Spark application using python then you will have to make changes 
            ```python
                .set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
            ```
            to
            ```python
                .set("spark.executor.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
            ```
            you will have to set the path to the log_dir like this dynamically
            ```python
                log_dir = os.path.join(project_dir, "log4j_properties", "logs")
                os.makedirs(log_dir, exist_ok=True)  # ensure it exists
            ```
        - In log4j.properties file you will have to change these things also 
            ```python
                log4j.appender.warnErrorFile.File=${custom.log.dir}/warn-error.log

                log4j.appender.debugFile.File=${custom.log.dir}/debug.log
            ```
- ```.set("spark.files", log4j_config_path)```
    - This ensures that **the Log4j.properties** file is distributed to all executors (workers) when the job starts.
    - This ensures that the log4j.properties file is distributed to all executors (workers) when the job starts.
    - In cluster mode, executors run on other nodes — they can’t see your local filesystem.
        - Setting spark.files tells Spark:
            - “Ship this file to every executor node.”
            - Then inside executors, Spark automatically adds the file to their working directory.
### How to configure JVM variables?
- Spark has a complex mechanism to read configuration settings. - Every spark application will look for a SPARK_HOME environment variable ```SPARK_HOME/conf/spark-defaults.conf```.
- If you have SPARK_HOME configured then spark will look for the conf directory of your SPARK_HOME.
- Spark is a JVM based application. It is written in scala and ir runs in a JAVA virtual machine.

### Why Log4j instead of the standard python logger.
We use Log4j instead of the standard python logger because collecting python log files is not integrated with Spark. Spark is designed to work with Log4j and most of the cluster manager also won't give you any issue when managing the Log4j files since it is well supported.
<br>
You can still use the python logger to send log messages to the  console. However if you want to collect your python logs to a central location then will have to configure the remote log handlers and use them in your pyspark program.
<br>
Setting up and using python remote log handlers could be an unnecessary complexity.




## First Spark program

In [20]:
from pyspark.sql import SparkSession

# Log4j related imports 
from pyspark import SparkConf
import os

# Determine if the code is running in a notebook or as a script
try:
    project_dir = os.path.dirname(os.path.abspath(__file__))
except:
    project_dir = os.getcwd()

# Dynamically define the log file directory 
log_dir = os.path.join(project_dir, "log4j_properties", "logs")
# This will create the log directories as per requirement if the directory does not exists
os.makedirs(log_dir,exist_ok=True)

# Set the path to the log4j.properties file where the configurations to the log4j logger is kept for Spark to use
log4j_config_path = os.path.join(project_dir, "log4j_properties","log4j.properties")

# Setting up the Spark configurations for Log4j
config = (
    SparkConf()
    .setAppName("TestApp")
    .setMaster("local[*]")
    # Supplying the directories where the logs must be generated to Spark from python instead of supplying it from log4j.properties file
    .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
    .set("spark.executor.extraJavaOptions",f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
    # Making sure all the executors also get the log4j.properties file
    .set("spark.files",log4j_config_path)
)

spark = SparkSession.builder.config(conf=config).getOrCreate()

dataset_file_path = os.path.join(project_dir, "dataset")

spark_df = spark.read.format("csv").option("headers","true").option("inferschema","true").load(f"{dataset_file_path}/sf-fire-calls.csv")

spark_df.show()

+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------------+--------------------+--------------------+------------------+--------------------+--------------------+-------------+---------+
|       _c0|   _c1|           _c2|             _c3|       _c4|       _c5|                 _c6|                 _c7|                 _c8| _c9|   _c10|     _c11|       _c12|_c13|            _c14|    _c15|         _c16|   _c17|         _c18|     _c19|          _c20|                _c21|                _c22|              _c23|                _c24|                _c25|         _c26|     _c27|
+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+--

### Explaination : 
- ```SparkSession`` here is essentially your Spark driver`
    - SparkSession is a singleton object so each spark application can have one and only one active SparkSession.
    - And if you look closely this do make sense because the SparkSession is your driver and you cannot have more than one driver in a spark application.


In [23]:
spark.stop()